# Territorial Control — Model Comparison Report

**Models compared:** Focal Mask (all features) · Random (proportional) · No-change (persistence)  
**Evaluation window:** validation months (`year_month > TRAIN_CUTOFF = 2025-06`)  

The report is split into two mirrored sections:
- **A — Overall:** all labelled cell-months in the validation set  
- **B — Transition cells:** only cell-months where the ground-truth label changed from the previous month  

Figures are saved to `src/reporting/`.

In [ ]:
import sys
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.metrics import (
    classification_report, confusion_matrix, f1_score
)
from IPython.display import display

# ── Paths ──────────────────────────────────────────────────────────────────────
PROJECT_ROOT  = Path('../..').resolve()
MODELS_DIR    = PROJECT_ROOT / 'models'
REPORTING_DIR = PROJECT_ROOT / 'src' / 'reporting'
REPORTING_DIR.mkdir(parents=True, exist_ok=True)

PRED_PATHS = {
    'focal_mask'        : MODELS_DIR / 'focal_mask'                  / 'predictions.csv',
    'random_proportion' : MODELS_DIR / 'baseline_random_proportion'  / 'predictions.csv',
    'no_change'         : MODELS_DIR / 'baseline_no_change'          / 'predictions.csv',
}

# ── Constants ──────────────────────────────────────────────────────────────────
TRAIN_CUTOFF = '2025-06'
CLASS_NAMES  = ['gov', 'opo', 'uncertain']
LABEL_MAP    = {'gov': 0, 'opo': 1, 'uncertain': 2}
SEED         = 20269999

MODEL_LABELS = {
    'focal_mask'        : 'Focal Mask',
    'random_proportion' : 'Random (proportional)',
    'no_change'         : 'No-change (persistence)',
}
MODEL_COLORS = {
    'focal_mask'        : '#2266CC',
    'random_proportion' : '#AAAAAA',
    'no_change'         : '#22BB44',
}

BG = '#F8F6F0'
plt.rcParams.update({'figure.facecolor': BG, 'axes.facecolor': BG})

In [ ]:
# ── Load predictions ───────────────────────────────────────────────────────────
dfs = {}
for name, path in PRED_PATHS.items():
    if not path.exists():
        raise FileNotFoundError(
            f'Predictions not found: {path}\n'
            'Run focal_mask_train.py / baseline scripts before this notebook.'
        )
    df = pd.read_csv(path)
    df['year_month'] = df['year_month'].astype(str)
    df['true_int']   = df['true_class'].map(LABEL_MAP)
    df['pred_int']   = df['pred_class'].map(LABEL_MAP)
    dfs[name] = df
    n_val = (~df['is_train_month']).sum()
    print(f'{name:<22}  total={len(df):,}  val={n_val:,}')

# ── Validate key alignment ─────────────────────────────────────────────────────
key_sets = {
    n: frozenset(zip(d['priogrid_gid'], d['year_month']))
    for n, d in dfs.items()
}
ref_name, ref_keys = 'focal_mask', key_sets['focal_mask']
for name, keys in key_sets.items():
    if name == ref_name:
        continue
    diff = (ref_keys - keys) | (keys - ref_keys)
    if diff:
        raise ValueError(
            f'Key mismatch: {ref_name} vs {name} — {len(diff)} differing pairs.\n'
            'Re-run the baselines with build_valid_gids() to realign.'
        )

print(f'\nKey alignment OK — all 3 files share {len(ref_keys):,} (gid, month) pairs.')

In [ ]:
def build_transition_keys(any_df):
    """
    Return a frozenset of (priogrid_gid, year_month) pairs in the val set
    where true_class at t differs from true_class at t-1.

    The shift is computed over the full time series (train + val), so the first
    val month looks back at the last training month of each cell — no cell-month
    is excluded from transition detection.
    Cells with no prior state are treated as non-transition (stable).
    """
    df = any_df[['priogrid_gid', 'year_month', 'true_class', 'is_train_month']].copy()
    df = df.sort_values(['priogrid_gid', 'year_month']).reset_index(drop=True)
    df['prev_true'] = df.groupby('priogrid_gid')['true_class'].shift(1)

    val = df[~df['is_train_month']].copy()
    # First appearance in the dataset (no train history): treat as stable
    val['prev_true'] = val['prev_true'].fillna(val['true_class'])
    val['is_transition'] = val['true_class'] != val['prev_true']

    n_trans = val['is_transition'].sum()
    n_total = len(val)
    print(f'Transition cell-months: {n_trans:,} / {n_total:,} ({100*n_trans/n_total:.1f}%)')
    print(val[val['is_transition']]['true_class'].value_counts().rename('transition targets'))

    return frozenset(
        zip(
            val.loc[val['is_transition'], 'priogrid_gid'],
            val.loc[val['is_transition'], 'year_month'],
        )
    )


transition_keys = build_transition_keys(dfs['focal_mask'])

In [ ]:
# ── Metric helpers (mirror evaluate_baselines.py) ──────────────────────────────

def get_val(df, trans_keys=None):
    """Val rows; if trans_keys given, restrict to those (priogrid_gid, year_month) pairs."""
    v = df[~df['is_train_month']].copy()
    if trans_keys is not None:
        mask = pd.Series(
            list(zip(v['priogrid_gid'], v['year_month'])), index=v.index
        ).isin(trans_keys)
        v = v[mask]
    return v


def compute_metrics(df_subset):
    """Classification report + confusion matrix for any val subset."""
    y_true  = df_subset['true_int'].values
    y_pred  = df_subset['pred_int'].values
    report  = classification_report(
        y_true, y_pred, target_names=CLASS_NAMES, output_dict=True, zero_division=0
    )
    cm      = confusion_matrix(y_true, y_pred, labels=[0, 1, 2])
    cm_norm = cm.astype(float) / (cm.sum(axis=1, keepdims=True) + 1e-8)
    return {
        'report'  : report,
        'cm'      : cm,
        'cm_norm' : cm_norm,
        'acc'     : (y_true == y_pred).mean(),
        'n'       : len(df_subset),
    }


def monthly_macro_f1(df, trans_keys=None):
    """Macro F1 for each val month, optionally restricted to transition cells."""
    val  = get_val(df, trans_keys)
    rows = []
    for month in sorted(val['year_month'].unique()):
        m = val[val['year_month'] == month]
        if len(m) == 0:
            continue
        rows.append({
            'year_month': month,
            'macro_f1'  : f1_score(m['true_int'], m['pred_int'],
                                   average='macro', zero_division=0),
            'n'         : len(m),
        })
    return pd.DataFrame(rows)


def headline_table(results):
    """Summary DataFrame for display."""
    rows = []
    for name, m in results.items():
        rep = m['report']
        rows.append({
            'Model'   : MODEL_LABELS[name],
            'N'       : m['n'],
            'Accuracy': f"{m['acc']:.1%}",
            'Macro F1': f"{rep['macro avg']['f1-score']:.1%}",
            'Gov F1'  : f"{rep['gov']['f1-score']:.1%}",
            'Opo F1'  : f"{rep['opo']['f1-score']:.1%}",
            'Unc F1'  : f"{rep['uncertain']['f1-score']:.1%}",
        })
    return pd.DataFrame(rows).set_index('Model')


# ── Figure helpers ─────────────────────────────────────────────────────────────

def plot_confusion_matrices(results, title_suffix='', save_path=None):
    """Three row-normalised confusion matrices, same color scale."""
    fig, axes = plt.subplots(1, 3, figsize=(18, 5), facecolor=BG)
    fig.suptitle(f'Confusion matrices — {title_suffix}', fontsize=13, fontweight='bold')

    for ax, (name, m) in zip(axes, results.items()):
        im = ax.imshow(m['cm_norm'], cmap='Blues', vmin=0, vmax=1, aspect='auto')
        ax.set_xticks(range(3)); ax.set_yticks(range(3))
        ax.set_xticklabels(CLASS_NAMES, fontsize=9)
        ax.set_yticklabels(CLASS_NAMES, fontsize=9)
        ax.set_xlabel('Predicted', fontsize=9); ax.set_ylabel('True', fontsize=9)
        ax.set_title(MODEL_LABELS[name], fontsize=10, fontweight='bold')
        ax.set_facecolor(BG)
        for i in range(3):
            for j in range(3):
                v = m['cm_norm'][i, j]; n = m['cm'][i, j]
                color = 'white' if v > 0.55 else '#222222'
                ax.text(j, i, f'{v:.2f}\n({n})', ha='center', va='center',
                        fontsize=8, color=color)

    fig.colorbar(im, ax=axes[-1], fraction=0.046, pad=0.04, label='Row-normalised recall')
    plt.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()


def plot_f1_bars(results, title_suffix='', save_path=None):
    """Grouped bar chart: F1 per class, one group of bars per class."""
    n_models = len(results)
    x        = np.arange(len(CLASS_NAMES))
    width    = 0.22
    offsets  = np.linspace(
        -(n_models - 1) / 2 * width,
         (n_models - 1) / 2 * width,
        n_models,
    )

    fig, ax = plt.subplots(figsize=(10, 5), facecolor=BG)
    ax.set_facecolor(BG)
    fig.suptitle(f'Per-class F1 — {title_suffix}', fontsize=12, fontweight='bold')

    for (name, m), offset in zip(results.items(), offsets):
        vals = [m['report'][cls]['f1-score'] for cls in CLASS_NAMES]
        bars = ax.bar(x + offset, vals, width,
                      label=MODEL_LABELS[name], color=MODEL_COLORS[name],
                      edgecolor='white', linewidth=0.5, alpha=0.9)
        for bar, v in zip(bars, vals):
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
                    f'{v:.0%}', ha='center', va='bottom', fontsize=8)

    ax.set_xticks(x)
    ax.set_xticklabels(['Government', 'Opposition', 'Uncertain'], fontsize=10)
    ax.set_ylim(0, 1.18); ax.set_ylabel('F1 score')
    ax.legend(fontsize=9); ax.grid(axis='y', alpha=0.3); ax.set_axisbelow(True)
    plt.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()


def plot_temporal_f1(dfs_dict, trans_keys=None, title_suffix='', save_path=None):
    """Monthly macro F1 line chart; one line per model."""
    fig, ax = plt.subplots(figsize=(12, 5), facecolor=BG)
    ax.set_facecolor(BG)
    fig.suptitle(f'Macro F1 by month — {title_suffix}', fontsize=12, fontweight='bold')

    for name, df in dfs_dict.items():
        mf1 = monthly_macro_f1(df, trans_keys)
        if mf1.empty:
            continue
        ax.plot(
            mf1['year_month'], mf1['macro_f1'],
            marker='o', lw=2, ms=5,
            label=MODEL_LABELS[name],
            color=MODEL_COLORS[name],
        )

    ax.set_xlabel('Month'); ax.set_ylabel('Macro F1')
    ax.set_ylim(-0.05, 1.05)
    ax.tick_params(axis='x', rotation=45)
    ax.legend(fontsize=9); ax.grid(alpha=0.3); ax.set_axisbelow(True)
    plt.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()

---
## A — Overall evaluation (all labelled val cells)

All 2,562 cell-months in the validation window, regardless of whether control
changed from the previous month.

In [ ]:
results_A = {
    name: compute_metrics(get_val(df))
    for name, df in dfs.items()
}

### A1 — Headline metrics

In [ ]:
display(headline_table(results_A))

The no-change model achieves the highest overall accuracy (97.7 %) and macro F1 —
a direct consequence of Myanmar territorial control being highly stable: roughly
97.7 % of val cell-months carry the same label as the previous month, so always
predicting "nothing changes" is almost always correct.

The focal mask trades raw aggregate performance for class-level discrimination,
especially on the rarer *opposition* and *uncertain* labels. The random baseline
lags both on every metric.

### A2 — Confusion matrices

In [ ]:
plot_confusion_matrices(
    results_A,
    title_suffix='Overall (all val cells)',
    save_path=REPORTING_DIR / 'A2_confusion_matrices_overall.png',
)

All matrices are row-normalised (diagonal = recall per class) on the same 0–1
colour scale.  
- The **no-change** diagonal is nearly perfect across all three classes — its
  errors are almost entirely cells that changed control, which it always
  mispredicts as stable.  
- The **focal mask** places a visible share of `gov` predictions into `uncertain`,
  reflecting genuine ambiguity the model hedges on; its `opo` recall is
  substantially better than the random baseline.  
- The **random** model's off-diagonal errors are proportional to class frequency,
  concentrating misses in the minority `opo` and `uncertain` columns.

### A3 — Per-class F1

In [ ]:
plot_f1_bars(
    results_A,
    title_suffix='Overall',
    save_path=REPORTING_DIR / 'A3_f1_bars_overall.png',
)

The random baseline's `gov` F1 is inflated relative to `opo` and `uncertain`
because sampling with training proportions (≈62 % government) biases most
predictions toward the majority class.  
The focal mask achieves meaningfully higher F1 on all three classes, and is the
only model with non-trivial `uncertain` F1.

### A4 — Temporal evolution of macro F1

In [ ]:
plot_temporal_f1(
    dfs,
    title_suffix='Overall',
    save_path=REPORTING_DIR / 'A4_temporal_f1_overall.png',
)

The no-change model's monthly F1 is flat and close to 1 throughout the
validation window — by construction, it can only fail in months where control
changes somewhere. The focal mask's trace is more volatile, tracking real
spatial-temporal variation in the conflict signal. The random baseline hovers
near its expected chance level.

**Key takeaways — Overall:**

- The no-change model's 97.7 % accuracy is a base-rate artefact: Myanmar territorial
  control is stable the vast majority of the time, so doing nothing is almost always
  right on aggregate.
- The focal mask outperforms the random baseline on every metric, but the gap with
  the persistence model on overall statistics makes the comparison misleading.
- The right test is not "who wins on all cells" but "who wins on the cells that
  actually changed" — which is exactly what Section B measures.

---
## B — Transition cells only

60 cell-months (2.3 % of the val set) where `true_class` at month *t* differs
from `true_class` at month *t*−1. For the first val month (2025-02), the
reference label is taken from the last training month (2025-01) of each cell.

This is the stress test: persistence is wrong by definition on every transition
cell, so only models that learn a genuine signal can exceed the random floor.

In [ ]:
results_B = {
    name: compute_metrics(get_val(df, transition_keys))
    for name, df in dfs.items()
}

for name, m in results_B.items():
    print(f'{MODEL_LABELS[name]:<28}  n={m["n"]}  acc={m["acc"]:.1%}')

In [ ]:
# ── Bootstrap CIs — transition cells only ────────────────────────────────────
# With per-class N as low as 4–23, no closed-form CI exists for AUC-PR / AUC-ROC.
# Bootstrap (n=1000, SEED-fixed) is the standard method; see Saito & Rehmsmeier
# (2015) and Hanley & McNeil (1982). Degenerate resamples (class absent) are
# skipped for that metric (excluded from nanpercentile) to avoid crashing.
# random_proportion has constant prob columns → AUC~0.5, AP~prevalence across
# all resamples; the trivially narrow CI reflects zero ranking variance by design.

from sklearn.metrics import (
    roc_auc_score, f1_score as _f1, average_precision_score as _ap
)
from sklearn.preprocessing import label_binarize as _lbin

N_BOOTSTRAP = 1000
_rng = np.random.default_rng(SEED)

def _bootstrap_trans(df, trans_keys, rng, n_boot=N_BOOTSTRAP):
    val   = get_val(df, trans_keys).reset_index(drop=True)
    n     = len(val)
    yt    = val['true_int'].values
    yp    = val['pred_int'].values
    yprob = val[['prob_gov', 'prob_opo', 'prob_uncertain']].values
    ybin  = _lbin(yt, classes=[0, 1, 2])

    # ── Point estimates ───────────────────────────────────────────────────────
    cls_data = {}
    for i, cls in enumerate(CLASS_NAMES):
        f1_pt = _f1(yt, yp, labels=[i], average=None, zero_division=0)
        f1_pt = float(f1_pt[0]) if len(f1_pt) else 0.0
        try:    auc_pt = roc_auc_score(ybin[:, i], yprob[:, i])
        except: auc_pt = float('nan')
        try:    ap_pt  = _ap(ybin[:, i], yprob[:, i])
        except: ap_pt  = float('nan')
        cls_data[cls] = {
            'n': int((yt == i).sum()),
            'f1_pt': f1_pt, 'auc_pt': auc_pt, 'ap_pt': ap_pt,
            'f1_b': [], 'auc_b': [], 'ap_b': [],
        }

    # ── Bootstrap loop ────────────────────────────────────────────────────────
    mac_f1_b, mac_auc_b, mac_ap_b = [], [], []
    for _ in range(n_boot):
        idx = rng.integers(0, n, size=n)
        yt_b = yt[idx]; yp_b = yp[idx]
        yprob_b = yprob[idx]; ybin_b = ybin[idx]

        per_f1, per_auc, per_ap = [], [], []
        for i, cls in enumerate(CLASS_NAMES):
            f1_b = _f1(yt_b, yp_b, labels=[i], average=None, zero_division=0)
            v_f1 = float(f1_b[0]) if len(f1_b) else 0.0
            cls_data[cls]['f1_b'].append(v_f1); per_f1.append(v_f1)

            pos, neg = ybin_b[:, i].sum(), (1 - ybin_b[:, i]).sum()
            if pos > 0 and neg > 0:
                try:    v = roc_auc_score(ybin_b[:, i], yprob_b[:, i])
                except: v = float('nan')
            else:
                v = float('nan')
            cls_data[cls]['auc_b'].append(v); per_auc.append(v)

            if pos > 0:
                try:    v = _ap(ybin_b[:, i], yprob_b[:, i])
                except: v = float('nan')
            else:
                v = float('nan')
            cls_data[cls]['ap_b'].append(v); per_ap.append(v)

        mac_f1_b.append(np.nanmean(per_f1))
        mac_auc_b.append(np.nanmean(per_auc))
        mac_ap_b.append(np.nanmean(per_ap))

    # ── Collect results ───────────────────────────────────────────────────────
    def _ci(arr): return (np.nanpercentile(arr, 2.5), np.nanpercentile(arr, 97.5))

    out = {}
    for cls in CLASS_NAMES:
        d = cls_data[cls]
        out[cls] = {
            'n':   d['n'],
            'f1':  (d['f1_pt'],  *_ci(d['f1_b'])),
            'auc': (d['auc_pt'], *_ci(d['auc_b'])),
            'ap':  (d['ap_pt'],  *_ci(d['ap_b'])),
        }
    mac_f1_pt  = np.nanmean([cls_data[c]['f1_pt']  for c in CLASS_NAMES])
    mac_auc_pt = np.nanmean([cls_data[c]['auc_pt'] for c in CLASS_NAMES])
    mac_ap_pt  = np.nanmean([cls_data[c]['ap_pt']  for c in CLASS_NAMES])
    out['macro'] = {
        'n':   n,
        'f1':  (mac_f1_pt,  *_ci(mac_f1_b)),
        'auc': (mac_auc_pt, *_ci(mac_auc_b)),
        'ap':  (mac_ap_pt,  *_ci(mac_ap_b)),
    }
    return out

print(f'Bootstrap n={N_BOOTSTRAP}, SEED={SEED}. Running for 3 models...')
boot_trans = {name: _bootstrap_trans(dfs[name], transition_keys, _rng)
              for name in dfs}
print('Done.')

def fmt_ci(triple, d=3):
    if any(np.isnan(v) for v in triple):
        return 'n/a'
    v, lo, hi = triple
    return f'{v:.{d}f} [{lo:.{d}f}, {hi:.{d}f}]'

# Summary N per class
ref = boot_trans['focal_mask']
print('\nTransition N per class:')
for key in CLASS_NAMES + ['macro']:
    print(f'  {key}: N={ref[key]["n"]}')

### B1 — Headline metrics

In [ ]:
# B1 — F1 table with N per class and 95% bootstrap CI
# Macro row first (most stable), then per-class rows.
rows = []
for cls_key in ['macro'] + CLASS_NAMES:
    for name in dfs:
        b = boot_trans[name]
        lbl = 'Macro' if cls_key == 'macro' else cls_key.capitalize()
        n   = b[cls_key]['n']
        rows.append({
            'Subset': f'{lbl} (N={n})',
            'Model' : MODEL_LABELS[name],
            'F1  value [95% CI]': fmt_ci(b[cls_key]['f1'], d=3),
        })

tbl = (
    pd.DataFrame(rows)
    .set_index(['Subset', 'Model'])
    .unstack('Model')
)
tbl.columns = tbl.columns.droplevel(0)
display(tbl)

The ranking flips completely relative to Section A.  
The no-change model scores **0 % accuracy** on transition cells — it always
predicts the previous state, which is by definition the wrong answer on a
cell that changed.  
The focal mask is the only model with non-trivial accuracy; the random baseline
provides the chance-level floor.

### B2 — Confusion matrices

In [ ]:
# B2 — Confusion matrices with per-class N in axis labels
n_variants = len(results_B)
fig, axes  = plt.subplots(1, n_variants, figsize=(7 * n_variants, 5), facecolor=BG)
if n_variants == 1:
    axes = [axes]
fig.suptitle('Row-normalised Confusion Matrices — Transition cells only', fontsize=13, fontweight='bold')

for ax, (name, m) in zip(axes, results_B.items()):
    mf1 = m['report']['macro avg']['f1-score']
    acc = m['acc']
    cls_n = [int(m['cm'][i].sum()) for i in range(3)]   # true class counts

    im = ax.imshow(m['cm_norm'], cmap='Blues', vmin=0, vmax=1, aspect='auto')
    ax.set_xticks(range(3)); ax.set_yticks(range(3))
    ax.set_xticklabels(CLASS_NAMES, fontsize=9)
    ax.set_yticklabels([f'{c}\n(N={cls_n[i]})' for i, c in enumerate(CLASS_NAMES)], fontsize=9)
    ax.set_xlabel('Predicted', fontsize=9); ax.set_ylabel('True', fontsize=9)
    ax.set_title(f'{MODEL_LABELS[name]}\nmacro F1={mf1:.3f}  acc={acc:.1%}  n_total={m["n"]}',
                 fontsize=10, fontweight='bold')
    ax.set_facecolor(BG)
    for i in range(3):
        for j in range(3):
            v = m['cm_norm'][i, j]; n_cell = m['cm'][i, j]
            color = 'white' if v > 0.55 else '#222222'
            ax.text(j, i, f'{v:.2f}\n({n_cell})', ha='center', va='center',
                    fontsize=8, color=color)

fig.colorbar(im, ax=axes[-1], fraction=0.046, pad=0.04, label='Row-normalised recall')
plt.tight_layout()
fig.savefig(REPORTING_DIR / 'B2_confusion_matrices_transitions.png', dpi=150, bbox_inches='tight')
plt.show()

The no-change confusion matrix has an **empty diagonal** — every transition is
classified into the wrong (previous) state, concentrated in the off-diagonal
blocks corresponding to the direction of each change.  
The focal mask achieves non-zero recall across all three target classes, though
the sample is small (n = 60) and the signal is noisy. The random baseline
distributes errors roughly proportionally to class frequency.

### B3 — Per-class F1

In [ ]:
# B3 — Per-class F1 with 95% bootstrap CI error bars
n_models = len(results_B)
x        = np.arange(len(CLASS_NAMES))
width    = 0.22
offsets  = np.linspace(-(n_models - 1) / 2 * width, (n_models - 1) / 2 * width, n_models)

fig, ax = plt.subplots(figsize=(10, 5), facecolor=BG)
ax.set_facecolor(BG)
fig.suptitle('Per-class F1 with 95% bootstrap CI — Transition cells only',
             fontsize=12, fontweight='bold')

for (name, m), offset in zip(results_B.items(), offsets):
    vals    = [m['report'][cls]['f1-score'] for cls in CLASS_NAMES]
    ci_lo   = [boot_trans[name][cls]['f1'][1] for cls in CLASS_NAMES]
    ci_hi   = [boot_trans[name][cls]['f1'][2] for cls in CLASS_NAMES]
    yerr_lo = [max(0.0, vals[i] - ci_lo[i]) for i in range(3)]
    yerr_hi = [max(0.0, ci_hi[i] - vals[i]) for i in range(3)]

    bars = ax.bar(x + offset, vals, width,
                  label=MODEL_LABELS[name], color=MODEL_COLORS[name],
                  edgecolor='white', linewidth=0.5, alpha=0.9)
    ax.errorbar(x + offset, vals, yerr=[yerr_lo, yerr_hi],
                fmt='none', color='#333333', capsize=3, linewidth=1.5, zorder=5)
    for i, (bar, v) in enumerate(zip(bars, vals)):
        top = v + yerr_hi[i] + 0.02
        ax.text(bar.get_x() + bar.get_width() / 2, top,
                f'{v:.0%}', ha='center', va='bottom', fontsize=7.5)

# N per class annotation on x-axis
ref_n = [boot_trans['focal_mask'][cls]['n'] for cls in CLASS_NAMES]
ax.set_xticks(x)
ax.set_xticklabels([f'{c.capitalize()}\n(N={ref_n[i]})' for i, c in enumerate(CLASS_NAMES)],
                   fontsize=10)
ax.set_ylim(0, 1.25); ax.set_ylabel('F1 score')
ax.legend(fontsize=9); ax.grid(axis='y', alpha=0.3); ax.set_axisbelow(True)
plt.tight_layout()
fig.savefig(REPORTING_DIR / 'B3_f1_bars_transitions.png', dpi=150, bbox_inches='tight')
plt.show()

The no-change model's F1 is 0 for every class because it never predicts the
post-transition state.  
The focal mask exceeds the random baseline on `gov` and `opo` transitions,
which are the most common change directions; performance on the rare `uncertain`
transitions is harder to interpret given the small count.

### B4 — Temporal evolution of macro F1 (transition cells)

In [ ]:
plot_temporal_f1(
    dfs,
    trans_keys=transition_keys,
    title_suffix='Transition cells only',
    save_path=REPORTING_DIR / 'B4_temporal_f1_transitions.png',
)

Monthly F1 on transitions is very volatile because the per-month sample is tiny
(0–9 transition cells per month). The no-change line is flat at 0 throughout.
The focal mask trace fluctuates but stays above the random baseline in most months,
suggesting the model learns some genuine spatial-temporal signal about where
transitions occur rather than just getting lucky on individual months.

**Key takeaways — Transition cells:**

- The no-change persistence model achieves exactly **0 %** on every transition
  metric — confirming it cannot detect any control change by construction.
- The focal mask is the only model with genuine predictive signal on transitions;
  the random baseline defines the lower bound any useful model must clear.
- Transition cells are rare (2.3 % of val) but represent the strategically
  important events in territorial control — a model that only optimises for
  overall accuracy will miss them entirely.

---
## C — Discrimination curves (PR & ROC, one-vs-rest)

Mirrors Sections A and B: each curve type is shown for the **overall** val set
and then restricted to **transition cells only**.

**How each model is represented:**

| Model | Representation |
|---|---|
| **Focal Mask** | Real softmax probabilities (`prob_gov / prob_opo / prob_uncertain`) |
| **No-change** | Hard scores (1.0 in predicted class, 0 elsewhere) — 3-point degenerate curve |
| **Random (proportional)** | Theoretical chance line: horizontal at class prevalence for PR; diagonal for ROC |

The random baseline is **not** drawn from its sampled predictions or constant
probability columns. A constant score vector has no ranking power and produces a
single operating point with sampling noise — the informative reference is the
theoretical limit: horizontal at class prevalence for PR (Saito & Rehmsmeier 2015)
and diagonal for ROC (Davis & Goadrich 2006).

In [ ]:
from sklearn.metrics import (
    roc_curve, auc, precision_recall_curve, average_precision_score,
)
from sklearn.preprocessing import label_binarize

PROB_COLS = ['prob_gov', 'prob_opo', 'prob_uncertain']

# Baseline de chance segundo Saito & Rehmsmeier (2015), "The Precision-Recall Plot
# Is More Informative than the ROC Plot When Evaluating Binary Classifiers on
# Imbalanced Datasets", PLOS ONE; e Davis & Goadrich (2006), "The Relationship
# Between Precision-Recall and ROC Curves", ICML. A PR de um classificador sem
# skill é a linha horizontal na prevalência (AUC-PR = prevalência); a ROC é a
# diagonal (AUC-ROC = 0.5). Para o random_proportion usamos essas linhas teóricas:
# os scores constantes armazenados no predictions.csv não têm poder discriminativo
# e produzem uma curva degenerada (ponto único) sem informação adicional.

def compute_curves(df, trans_keys=None):
    """
    One-vs-rest PR and ROC from probability columns.
    trans_keys: if given, restrict to transition cell-months only.
    Works for focal_mask (smooth softmax probs) and no_change (hard 0/1 scores).
    """
    val    = get_val(df, trans_keys)
    y_true = val['true_int'].values
    y_prob = val[PROB_COLS].values
    y_bin  = label_binarize(y_true, classes=[0, 1, 2])

    pr_data, roc_data = {}, {}
    for i, cls in enumerate(CLASS_NAMES):
        prec, rec, _  = precision_recall_curve(y_bin[:, i], y_prob[:, i])
        ap             = average_precision_score(y_bin[:, i], y_prob[:, i])
        pr_data[cls]   = {'prec': prec, 'rec': rec, 'ap': ap}

        fpr, tpr, _    = roc_curve(y_bin[:, i], y_prob[:, i])
        roc_data[cls]  = {'fpr': fpr, 'tpr': tpr, 'auc': auc(fpr, tpr)}

    return {'pr': pr_data, 'roc': roc_data}


# Overall val set
curves_overall = {
    name: compute_curves(dfs[name])
    for name in ['focal_mask', 'no_change']
}

# Transition cells only
curves_trans = {
    name: compute_curves(dfs[name], transition_keys)
    for name in ['focal_mask', 'no_change']
}

# Prevalence per class — used as the AUC-PR for the random chance reference
val_ref            = get_val(dfs['focal_mask'])
prevalence_overall = {cls: (val_ref['true_int'].values == i).mean()
                      for i, cls in enumerate(CLASS_NAMES)}

trans_ref          = get_val(dfs['focal_mask'], transition_keys)
prevalence_trans   = {cls: (trans_ref['true_int'].values == i).mean()
                      for i, cls in enumerate(CLASS_NAMES)}

print('Overall prevalence:')
for cls, p in prevalence_overall.items():
    print(f'  {cls}: {p:.3f}')
print('\nTransition-cell prevalence:')
for cls, p in prevalence_trans.items():
    print(f'  {cls}: {p:.3f}')
print()
for subset, curves_dict in [('Overall', curves_overall), ('Transitions', curves_trans)]:
    print(f'{subset}:')
    for name in ['focal_mask', 'no_change']:
        line = f'  {MODEL_LABELS[name]}: '
        line += '  '.join(
            f'{cls} AP={curves_dict[name]["pr"][cls]["ap"]:.3f}'
            f'/AUC={curves_dict[name]["roc"][cls]["auc"]:.3f}'
            for cls in CLASS_NAMES
        )
        print(line)
    print()

### C — Overall: Precision-Recall curves

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5), facecolor=BG)
fig.suptitle('Precision-Recall curves — Overall (one-vs-rest, val set)',
             fontsize=13, fontweight='bold')

for ax, cls in zip(axes, CLASS_NAMES):
    ax.set_facecolor(BG)
    ax.set_title(f'Class: {cls}', fontsize=11, fontweight='bold')
    ax.set_xlabel('Recall', fontsize=9)
    ax.set_ylabel('Precision', fontsize=9)
    ax.set_xlim(-0.02, 1.05)
    ax.set_ylim(-0.02, 1.05)
    ax.grid(alpha=0.3)
    ax.set_axisbelow(True)

    d = curves_overall['focal_mask']['pr'][cls]
    ax.step(d['rec'], d['prec'], where='post',
            color=MODEL_COLORS['focal_mask'], lw=2,
            label=f"Focal Mask (AP={d['ap']:.3f})")

    d = curves_overall['no_change']['pr'][cls]
    ax.step(d['rec'], d['prec'], where='post',
            color=MODEL_COLORS['no_change'], lw=2, ls='--',
            label=f"No-change (AP={d['ap']:.3f})")

    # Saito & Rehmsmeier (2015): PR of a no-skill classifier = horizontal at prevalence
    prev = prevalence_overall[cls]
    ax.axhline(prev, color=MODEL_COLORS['random_proportion'], lw=1.5, ls=':',
               label=f"Random chance (AP={prev:.3f})")
    ax.fill_between([0, 1], prev, alpha=0.07, color=MODEL_COLORS['random_proportion'])

    ax.legend(fontsize=8, loc='upper right')

plt.tight_layout()
fig.savefig(REPORTING_DIR / 'C1_pr_overall.png', dpi=150, bbox_inches='tight')
plt.show()

The no-change model's high overall AP (0.978 / 0.938 / 0.939) reflects temporal
persistence, not discriminative skill: it operates at a single hard threshold,
so the curve degenerates to one operating point. Its precision is high because the
inherited state is almost always correct on stable cells.

The focal mask traces a proper curve across the full recall range, with AP
substantially above the random-chance floor (grey region) for all three classes.

### C — Overall: ROC curves

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5), facecolor=BG)
fig.suptitle('ROC curves — Overall (one-vs-rest, val set)',
             fontsize=13, fontweight='bold')

for ax, cls in zip(axes, CLASS_NAMES):
    ax.set_facecolor(BG)
    ax.set_title(f'Class: {cls}', fontsize=11, fontweight='bold')
    ax.set_xlabel('False Positive Rate', fontsize=9)
    ax.set_ylabel('True Positive Rate', fontsize=9)
    ax.set_xlim(-0.02, 1.02)
    ax.set_ylim(-0.02, 1.02)
    ax.grid(alpha=0.3)
    ax.set_axisbelow(True)

    d = curves_overall['focal_mask']['roc'][cls]
    ax.plot(d['fpr'], d['tpr'],
            color=MODEL_COLORS['focal_mask'], lw=2,
            label=f"Focal Mask (AUC={d['auc']:.3f})")

    d = curves_overall['no_change']['roc'][cls]
    ax.plot(d['fpr'], d['tpr'],
            color=MODEL_COLORS['no_change'], lw=2, ls='--',
            label=f"No-change (AUC={d['auc']:.3f})")

    # Davis & Goadrich (2006): ROC of a no-skill classifier = diagonal
    ax.plot([0, 1], [0, 1],
            color=MODEL_COLORS['random_proportion'], lw=1.5, ls=':',
            label='Random chance (AUC=0.500)')
    ax.fill_between([0, 1], [0, 1], alpha=0.07, color=MODEL_COLORS['random_proportion'])

    ax.legend(fontsize=8, loc='lower right')

plt.tight_layout()
fig.savefig(REPORTING_DIR / 'C2_roc_overall.png', dpi=150, bbox_inches='tight')
plt.show()

The no-change model's apparent AUC (0.977–0.985) is also a persistence artefact:
scoring 1.0 for the inherited class means false positives are rare when the predicted
class is the dominant one. The degenerate shape (three points, two line segments)
makes clear there is no actual ranking — the model has only one operating point.

The focal mask's ROC AUC (0.872–0.926) is lower than the no-change model's, yet
substantially more informative: it reflects genuine soft discrimination across
2,562 distinct score values.

### C — Transition cells: Precision-Recall curves

On transition cell-months the no-change model's scores are structurally inverted:
it assigns 1.0 to the *previous* class (the one cells are leaving) and 0.0 to the
*new* class (the true target). The PR and ROC curves on this subset therefore
expose whether models can discriminate at all when the ground truth changes.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5), facecolor=BG)
fig.suptitle('Precision-Recall curves — Transition cells only (one-vs-rest)',
             fontsize=13, fontweight='bold')

for ax, cls in zip(axes, CLASS_NAMES):
    n_cls = boot_trans['focal_mask'][cls]['n']
    ax.set_facecolor(BG)
    ax.set_title(f'Class: {cls}  (N={n_cls})', fontsize=11, fontweight='bold')
    ax.set_xlabel('Recall', fontsize=9)
    ax.set_ylabel('Precision', fontsize=9)
    ax.set_xlim(-0.02, 1.05)
    ax.set_ylim(-0.02, 1.05)
    ax.grid(alpha=0.3)
    ax.set_axisbelow(True)

    d = curves_trans['focal_mask']['pr'][cls]
    ax.step(d['rec'], d['prec'], where='post',
            color=MODEL_COLORS['focal_mask'], lw=2,
            label=f"Focal Mask (AP={d['ap']:.3f})")

    d = curves_trans['no_change']['pr'][cls]
    ax.step(d['rec'], d['prec'], where='post',
            color=MODEL_COLORS['no_change'], lw=2, ls='--',
            label=f"No-change (AP={d['ap']:.3f})")

    prev = prevalence_trans[cls]
    ax.axhline(prev, color=MODEL_COLORS['random_proportion'], lw=1.5, ls=':',
               label=f"Random chance (AP={prev:.3f})")
    ax.fill_between([0, 1], prev, alpha=0.07, color=MODEL_COLORS['random_proportion'])

    ax.legend(fontsize=8, loc='upper right')

plt.tight_layout()
fig.savefig(REPORTING_DIR / 'C3_pr_transitions.png', dpi=150, bbox_inches='tight')
plt.show()

The no-change model's AP on transition cells equals the random-chance floor
(≈ class prevalence) for all three classes. This is not a coincidence: because
it assigns 0.0 to every true-class probability on transition cells, it has
no ranking signal — only a constant score remains, which produces AP = prevalence
by definition.

The focal mask's AP on transitions (gov ≈ 0.47, opo ≈ 0.43, uncertain ≈ 0.15)
is modest and near chance on this n=60 subset, but the curve traces a real
precision-recall trade-off rather than collapsing to a point.

### C — Transition cells: ROC curves

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5), facecolor=BG)
fig.suptitle('ROC curves — Transition cells only (one-vs-rest)',
             fontsize=13, fontweight='bold')

for ax, cls in zip(axes, CLASS_NAMES):
    n_cls = boot_trans['focal_mask'][cls]['n']
    ax.set_facecolor(BG)
    ax.set_title(f'Class: {cls}  (N={n_cls})', fontsize=11, fontweight='bold')
    ax.set_xlabel('False Positive Rate', fontsize=9)
    ax.set_ylabel('True Positive Rate', fontsize=9)
    ax.set_xlim(-0.02, 1.02)
    ax.set_ylim(-0.02, 1.02)
    ax.grid(alpha=0.3)
    ax.set_axisbelow(True)

    d = curves_trans['focal_mask']['roc'][cls]
    ax.plot(d['fpr'], d['tpr'],
            color=MODEL_COLORS['focal_mask'], lw=2,
            label=f"Focal Mask (AUC={d['auc']:.3f})")

    d = curves_trans['no_change']['roc'][cls]
    ax.plot(d['fpr'], d['tpr'],
            color=MODEL_COLORS['no_change'], lw=2, ls='--',
            label=f"No-change (AUC={d['auc']:.3f})")

    ax.plot([0, 1], [0, 1],
            color=MODEL_COLORS['random_proportion'], lw=1.5, ls=':',
            label='Random chance (AUC=0.500)')
    ax.fill_between([0, 1], [0, 1], alpha=0.07, color=MODEL_COLORS['random_proportion'])

    ax.legend(fontsize=8, loc='lower right')

plt.tight_layout()
fig.savefig(REPORTING_DIR / 'C4_roc_transitions.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# C — Transition cells: AUC-ROC and Average Precision with 95% bootstrap CI
# Macro is the primary summary; per-class rows show the uncertainty driving the macro.
# random_proportion's prob columns are constant → AUC bootstrap collapses to ~0.5
# and AP bootstrap collapses to ~prevalence, confirming zero ranking variance.

rows_auc, rows_ap = [], []
for cls_key in ['macro'] + CLASS_NAMES:
    for name in dfs:
        b  = boot_trans[name]
        lbl = 'Macro' if cls_key == 'macro' else cls_key.capitalize()
        n   = b[cls_key]['n']
        rows_auc.append({
            'Subset': f'{lbl} (N={n})',
            'Model' : MODEL_LABELS[name],
            'AUC-ROC  value [95% CI]': fmt_ci(b[cls_key]['auc'], d=3),
        })
        rows_ap.append({
            'Subset': f'{lbl} (N={n})',
            'Model' : MODEL_LABELS[name],
            'Avg Precision  value [95% CI]': fmt_ci(b[cls_key]['ap'], d=3),
        })

def _pivot(rows, col):
    return (
        pd.DataFrame(rows)
        .set_index(['Subset', 'Model'])
        .unstack('Model')
        .droplevel(0, axis=1)
    )

print('AUC-ROC (one-vs-rest, transition cells):')
display(_pivot(rows_auc, 'AUC-ROC  value [95% CI]'))
print('\nAverage Precision (one-vs-rest, transition cells):')
display(_pivot(rows_ap, 'Avg Precision  value [95% CI]'))

The no-change model's ROC falls **below the diagonal** for all three classes
(AUC: gov≈0.08, opo≈0.19, uncertain≈0.40). This is structural, not sampling noise:
it scores 1.0 for the class cells are *leaving*, so its high scores concentrate
on false positives. The bootstrap CI for no-change AUC on transitions will be
narrow and well below 0.5, confirming the anti-classifier behaviour is stable.

The focal mask's AUC and AP on transitions are near chance at the point estimate,
with wide bootstrap CIs — see the tables above. The macro AUC and AP are the
most reliable summary; per-class values should not be interpreted in isolation
given the small per-class N.

---

### Interpretive note on transition metrics

**These metrics indicate direction, not magnitude.** Three facts the reader must
keep in mind:

1. **Small N.** The total number of transition cell-months is ≈40–60; per-class
   counts range from roughly 4 to 23. This is the genuine size of the
   strategically relevant signal in the data.

2. **Wide CIs are honest.** The 95% bootstrap intervals are deliberately wide
   where the data are few. A CI of [0.10, 0.80] does not mean the estimator is
   wrong — it means the data cannot pin down the value, which is itself
   informative. Reporting a point estimate without the interval would
   misrepresent the state of evidence.

3. **The qualitative conclusion is robust despite the uncertainty.**  
   - The **no-change** model's F1 is exactly 0 on every transition class —
     a degenerate outcome that holds with probability 1 regardless of N,
     because the model always predicts the previous state.  
   - Its AUC falls *below* 0.5 on all three classes — also deterministic,
     not sampling noise: it actively misranks cells at transition time.  
   - The **focal mask** produces non-zero F1 and AUC above the anti-classifier
     floor for most classes, demonstrating that the learned representations
     carry some signal about impending control changes, even if the CI is
     too wide to quantify the effect size precisely.

The transition section is therefore best read as **model separation evidence**
(focal mask ≫ no-change, focal mask ≥ random) rather than as a precise
performance measurement. Larger evaluation windows or additional transition
events — outside the current 29-month panel — would be needed to narrow the CIs.

---
## Conclusion

The two sections expose a fundamental tension in evaluating territorial control
models on imbalanced, temporally persistent data:

| | Overall | Transitions only |
|---|---|---|
| **No-change (persistence)** | Best (97.7 % acc, 97.1 % macro F1) | Worst (0 % acc by construction) |
| **Focal Mask** | Second (71.9 % acc, 67.2 % macro F1) | Best (only model above chance) |
| **Random (proportional)** | Worst (45.4 % acc, 31.8 % macro F1) | Lower bound (chance level) |

**The no-change model's dominance on overall metrics is a base-rate effect,
not a capability.** Myanmar territorial control is stable in ≈97.7 % of
cell-months; inheriting the previous state is almost always correct in aggregate.
This makes overall accuracy and macro F1 uninformative as primary benchmarks.

**The relevant benchmark is transition detection.** On the 60 cell-months where
control actually shifted, the persistence model fails completely and the focal
mask is the only model that learns any signal above the random floor.

**Directions for improvement:**
- Increase the focal-mask loss weight on known transition regions to sharpen
  recall where it matters most.
- Add auxiliary features that capture conflict escalation dynamics
  (e.g., event-count acceleration, sudden actor-mix changes) as early signals
  of imminent control transfers.
- Explore a two-stage architecture: first classify *will this cell transition?*
  (binary), then predict the new state — allowing dedicated optimisation for
  each sub-problem.